# Tratamento e Limpeza de Dados de Comentários

Este notebook realiza limpeza e validação do dataset de comentários do YouTube, preparando-o para análises posteriores.

**Pipeline**: Filtragem por idioma → Validação de datas → Tratamento de valores ausentes → Remoção de colunas não informativas → Remoção de duplicatas

**Input**: `final_commments_match.csv` (comentários brutos)
**Output**: `final_comments_match_cleaned.csv` (comentários limpos e validados)

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from tqdm import tqdm
from transformers import pipeline
import time
from wordcloud import WordCloud
from nltk.corpus import stopwords
import nltk
import re
from scipy import stats
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from scipy.stats import zscore
import warnings

stopwords_pt = set(stopwords.words('portuguese'))

2026-01-06 18:16:01.585167: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


---
## 1. Carregamento e Filtragem por Idioma

Carrega dataset de comentários e filtra apenas vídeos classificados como português:
- **Fonte**: Lista de IDs de vídeos em português (`ids_portuguese.txt`) gerada por detecção automática de idioma
- **Redução**: Remove comentários de vídeos em outros idiomas para focar na análise de conteúdo em português

In [2]:
df = pd.read_csv('cleaned_data/final_commments_match.csv')
df.head()

/tmp/ipykernel_836/2894656101.py:1: DtypeWarning: Columns (9,11,12) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('cleaned_data/final_commments_match.csv')


,video_id,comment_id,author,author_profile_image_url,author_channel_url,author_channel_id,comment,published_at,updated_at,like_count,viewer_rating,can_rate,is_reply,parent_id,channel_id,language,language_langdetect
0,g3YDGLa2pWg,UgzP9hlUh9w_IWQksvN4AaABAg,@dr.gabrielfeitosa,https://yt3.ggpht.com/yp_y0N8quVmXxwhO3S1rQqlX...,http://www.youtube.com/@dr.gabrielfeitosa,UCYgV9EbxVSNdDHBr0GWLCmQ,Segue uma playlist sobre colaterais de hormôni...,2024-12-31 23:07:02+00:00,2024-12-31T23:07:02Z,1,none,True,False,NaN,UCYgV9EbxVSNdDHBr0GWLCmQ,pt,NaN
1,g3YDGLa2pWg,UgzP9hlUh9w_IWQksvN4AaABAg.ACjBEF6NnUrACpvH__rSZ6,@Fabio_Britto,https://yt3.ggpht.com/ytc/AIdro_lbzoKmUeO5NasE...,http://www.youtube.com/@Fabio_Britto,UCaHIZ0V18SSmxIPQbKRyf4w,"Por favor. Não fumo, não bebo a 2 anos, tinha ...",2025-01-03 13:53:38+00:00,2025-01-03T13:53:38Z,0,none,True,True,UgzP9hlUh9w_IWQksvN4AaABAg,UCYgV9EbxVSNdDHBr0GWLCmQ,pt,NaN
2,g3YDGLa2pWg,UgzNj3A5ReT4piuNYEN4AaABAg,@debyomental,https://yt3.ggpht.com/vuK55ebWZ0zCTt7q1tCiFAS6...,http://www.youtube.com/@debyomental,UCQsZB-Cd3kpnCjxbslnm1kg,Obrigado dr.gabriel! Eu sempre tive o sonho de...,2025-01-08 19:55:43+00:00,2025-01-08T19:55:43Z,0,none,True,False,NaN,UCYgV9EbxVSNdDHBr0GWLCmQ,pt,NaN
3,g3YDGLa2pWg,UgzNj3A5ReT4piuNYEN4AaABAg.AD2RgwaVf-aAD9OQUOJ1MM,@diegosilva3310,https://yt3.ggpht.com/ytc/AIdro_mZTUGWmVKXIHr5...,http://www.youtube.com/@diegosilva3310,UCu4uK6PpfnOegglK4TpStwA,"@@debyomental afere a pressao mano, monitora c...",2025-01-11 12:41:48+00:00,2025-01-11T12:41:48Z,0,none,True,True,UgzNj3A5ReT4piuNYEN4AaABAg,UCYgV9EbxVSNdDHBr0GWLCmQ,pt,NaN
4,g3YDGLa2pWg,UgyN15Q5w3bYXx73vuV4AaABAg,@ancientknight4653,https://yt3.ggpht.com/ytc/AIdro_nAn5L24RfZZyrv...,http://www.youtube.com/@ancientknight4653,UCZ6A0MP48-hUXNTf9EQGBqQ,Muito informativo. Desejo crescimento para o c...,2025-01-06 01:32:15+00:00,2025-01-06T01:32:15Z,0,none,True,False,NaN,UCYgV9EbxVSNdDHBr0GWLCmQ,pt,NaN


In [3]:
df.iloc[:5, [9, 11, 12]]

,like_count,can_rate,is_reply
0,1,True,False
1,0,True,True
2,0,True,False
3,0,True,True
4,0,True,False


In [4]:
df.shape

(552220, 17)

In [5]:
df['video_id'].nunique()

8615

In [6]:
with open('cleaned_data/ids_portuguese.txt', 'r') as f:
    ids = f.read().splitlines()

In [7]:
df = df[df['video_id'].isin(ids)]

In [8]:
df.shape

(534840, 17)

In [9]:
df['video_id'].nunique()

6769

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 534840 entries, 0 to 552219
Data columns (total 17 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   video_id                  534840 non-null  object
 1   comment_id                534840 non-null  object
 2   author                    529187 non-null  object
 3   author_profile_image_url  534840 non-null  object
 4   author_channel_url        534840 non-null  object
 5   author_channel_id         534840 non-null  object
 6   comment                   534840 non-null  object
 7   published_at              513677 non-null  object
 8   updated_at                513677 non-null  object
 9   like_count                513677 non-null  object
 10  viewer_rating             513677 non-null  object
 11  can_rate                  513677 non-null  object
 12  is_reply                  513677 non-null  object
 13  parent_id                 246600 non-null  object
 14  channel_i

In [11]:
df.loc[(df['language_langdetect'].notna()), ['language', 'language_langdetect']].head()

,language,language_langdetect
35,it,pt
239,sw,pt
241,it,pt
316,en,pt
1045,ur,pt


In [12]:
df.loc[(df['language'] != 'pt') & (df['language_langdetect'] != 'pt')].shape

(21163, 17)

In [13]:
df[(df['language_langdetect'].isna())].shape

(525319, 17)

In [14]:
df[df['language'] != 'pt'].shape

(30684, 17)

In [15]:
df[df['language_langdetect'] == df['language']].shape

(0, 17)

In [16]:
df.isna().sum()

video_id                         0
comment_id                       0
author                        5653
author_profile_image_url         0
author_channel_url               0
author_channel_id                0
comment                          0
published_at                 21163
updated_at                   21163
like_count                   21163
viewer_rating                21163
can_rate                     21163
is_reply                     21163
parent_id                   288240
channel_id                   21163
language                     21163
language_langdetect         525319
dtype: int64

In [17]:
df.head()

,video_id,comment_id,author,author_profile_image_url,author_channel_url,author_channel_id,comment,published_at,updated_at,like_count,viewer_rating,can_rate,is_reply,parent_id,channel_id,language,language_langdetect
0,g3YDGLa2pWg,UgzP9hlUh9w_IWQksvN4AaABAg,@dr.gabrielfeitosa,https://yt3.ggpht.com/yp_y0N8quVmXxwhO3S1rQqlX...,http://www.youtube.com/@dr.gabrielfeitosa,UCYgV9EbxVSNdDHBr0GWLCmQ,Segue uma playlist sobre colaterais de hormôni...,2024-12-31 23:07:02+00:00,2024-12-31T23:07:02Z,1,none,True,False,NaN,UCYgV9EbxVSNdDHBr0GWLCmQ,pt,NaN
1,g3YDGLa2pWg,UgzP9hlUh9w_IWQksvN4AaABAg.ACjBEF6NnUrACpvH__rSZ6,@Fabio_Britto,https://yt3.ggpht.com/ytc/AIdro_lbzoKmUeO5NasE...,http://www.youtube.com/@Fabio_Britto,UCaHIZ0V18SSmxIPQbKRyf4w,"Por favor. Não fumo, não bebo a 2 anos, tinha ...",2025-01-03 13:53:38+00:00,2025-01-03T13:53:38Z,0,none,True,True,UgzP9hlUh9w_IWQksvN4AaABAg,UCYgV9EbxVSNdDHBr0GWLCmQ,pt,NaN
2,g3YDGLa2pWg,UgzNj3A5ReT4piuNYEN4AaABAg,@debyomental,https://yt3.ggpht.com/vuK55ebWZ0zCTt7q1tCiFAS6...,http://www.youtube.com/@debyomental,UCQsZB-Cd3kpnCjxbslnm1kg,Obrigado dr.gabriel! Eu sempre tive o sonho de...,2025-01-08 19:55:43+00:00,2025-01-08T19:55:43Z,0,none,True,False,NaN,UCYgV9EbxVSNdDHBr0GWLCmQ,pt,NaN
3,g3YDGLa2pWg,UgzNj3A5ReT4piuNYEN4AaABAg.AD2RgwaVf-aAD9OQUOJ1MM,@diegosilva3310,https://yt3.ggpht.com/ytc/AIdro_mZTUGWmVKXIHr5...,http://www.youtube.com/@diegosilva3310,UCu4uK6PpfnOegglK4TpStwA,"@@debyomental afere a pressao mano, monitora c...",2025-01-11 12:41:48+00:00,2025-01-11T12:41:48Z,0,none,True,True,UgzNj3A5ReT4piuNYEN4AaABAg,UCYgV9EbxVSNdDHBr0GWLCmQ,pt,NaN
4,g3YDGLa2pWg,UgyN15Q5w3bYXx73vuV4AaABAg,@ancientknight4653,https://yt3.ggpht.com/ytc/AIdro_nAn5L24RfZZyrv...,http://www.youtube.com/@ancientknight4653,UCZ6A0MP48-hUXNTf9EQGBqQ,Muito informativo. Desejo crescimento para o c...,2025-01-06 01:32:15+00:00,2025-01-06T01:32:15Z,0,none,True,False,NaN,UCYgV9EbxVSNdDHBr0GWLCmQ,pt,NaN


In [18]:
df['published_at'] = pd.to_datetime(df['published_at'], utc=True, errors='coerce')
df['updated_at'] = pd.to_datetime(df['updated_at'], utc=True, errors='coerce')

---
## 2. Validação e Tratamento de Datas

Converte colunas de data para datetime e remove registros inválidos:
- **published_at / updated_at**: Conversão para formato datetime com timezone UTC
- **Filtro**: Remove comentários com datas em 2025 (ano incompleto)
- **Valores nulos**: Identifica e remove registros com datas ausentes ou inválidas

In [19]:
df[df['published_at'].isna() | df['updated_at'].isna()].shape

(21163, 17)

In [20]:
df.isna().sum()

video_id                         0
comment_id                       0
author                        5653
author_profile_image_url         0
author_channel_url               0
author_channel_id                0
comment                          0
published_at                 21163
updated_at                   21163
like_count                   21163
viewer_rating                21163
can_rate                     21163
is_reply                     21163
parent_id                   288240
channel_id                   21163
language                     21163
language_langdetect         525319
dtype: int64

In [21]:
df = df[(df['published_at'].dt.year != 2025) & (df['updated_at'].dt.year != 2025)]

In [22]:
df[df['published_at'].dt.year == 2025].shape

(0, 17)

In [23]:
# Como a coluna 'language_langdetect' só possui valores onde 'language' é diferente de 'pt', vou preencher os valores ausentes com 'rob_pt'
df.loc[df['language_langdetect'].isna() & (df['language'] == 'pt'), 'language_langdetect'] = 'rob_pt'

---
## 3. Tratamento de Valores Ausentes

Lida com dados faltantes de forma consistente:
- **language_langdetect**: Preenche com 'rob_pt' quando language='pt' (indica detecção por RoBERTa)
- **parent_id**: Preenche com 'root' para comentários principais (não-respostas)
- **Remoção**: Drop de linhas com valores ausentes críticos (datas, IDs) - ~26.850 comentários removidos
- **Validação**: Verifica consistência entre parent_id e is_reply

In [24]:
df.loc[df['parent_id'].isna(), ['parent_id', 'comment']].head()

,parent_id,comment
0,NaN,Segue uma playlist sobre colaterais de hormôni...
66,NaN,Excelente vídeo parabéns
67,NaN,"Tenho 52 anos 1,80 de altura com 95 kg, faço a..."
87,NaN,"Dr, o q o Sr acha de 1 ml/14 dias?"
93,NaN,Link dos Danones: https://whatsa.me/5533991157...


In [25]:
df.loc[(df['parent_id'].isna()) & (df['is_reply'] == False)].shape

(264618, 17)

In [26]:
# Todos os comentários que não possuem parent_id possuem is_reply como False
print('Total de comentários que não possuem parent_id:', df.loc[df['parent_id'].isna(), 'comment'].count())
print('Total de comentários que não possuem parent_id e não são respostas:', df.loc[(df['parent_id'].isna()) & (df['is_reply'] == False), 'comment'].count())
print('Total de comentários com informação ausente sobre is_reply:', df['is_reply'].isna().sum())
print('Diferença: ', df.loc[df['parent_id'].isna(), 'comment'].count() - df.loc[(df['parent_id'].isna()) & (df['is_reply'] == False), 'comment'].count())

Total de comentários que não possuem parent_id: 285781
Total de comentários que não possuem parent_id e não são respostas: 264618
Total de comentários com informação ausente sobre is_reply: 21163
Diferença:  21163


In [27]:
# Como havia um grande número de valores ausentes na coluna 'parent_id', assumi que esses comentários são raízes de árvores de comentários, então preenchi com 'root'
df.loc[(df['parent_id'].isna()) & (df['is_reply'] == False), 'parent_id'] = 'root'

In [28]:
df.isna().sum()

video_id                        0
comment_id                      0
author                       5528
author_profile_image_url        0
author_channel_url              0
author_channel_id               0
comment                         0
published_at                21163
updated_at                  21163
like_count                  21163
viewer_rating               21163
can_rate                    21163
is_reply                    21163
parent_id                   21163
channel_id                  21163
language                    21163
language_langdetect         21163
dtype: int64

In [29]:
df.loc[df['published_at'].isna()].head()

,video_id,comment_id,author,author_profile_image_url,author_channel_url,author_channel_id,comment,published_at,updated_at,like_count,viewer_rating,can_rate,is_reply,parent_id,channel_id,language,language_langdetect
437590,nCtyfB2qSRw,UgzNiQdoxnPl3oORAed4AaABAg,@jailtongiga,https://yt3.ggpht.com/b0M_lJEtjhRqpenO94EmyGx6...,http://www.youtube.com/@jailtongiga,UCJVjexEjkVqG6-YTEvzRvhA,Salve! gostaria contar minha história meu nome...,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
437591,nCtyfB2qSRw,UgzNiQdoxnPl3oORAed4AaABAg,@jailtongiga,https://yt3.ggpht.com/b0M_lJEtjhRqpenO94EmyGx6...,http://www.youtube.com/@jailtongiga,UCJVjexEjkVqG6-YTEvzRvhA,Salve! gostaria contar minha história meu nome...,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
437592,nCtyfB2qSRw,UgzNiQdoxnPl3oORAed4AaABAg,@jailtongiga,https://yt3.ggpht.com/b0M_lJEtjhRqpenO94EmyGx6...,http://www.youtube.com/@jailtongiga,UCJVjexEjkVqG6-YTEvzRvhA,Salve! gostaria contar minha história meu nome...,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
437593,nCtyfB2qSRw,UgzNiQdoxnPl3oORAed4AaABAg,@jailtongiga,https://yt3.ggpht.com/b0M_lJEtjhRqpenO94EmyGx6...,http://www.youtube.com/@jailtongiga,UCJVjexEjkVqG6-YTEvzRvhA,Salve! gostaria contar minha história meu nome...,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
437594,nCtyfB2qSRw,UgzNiQdoxnPl3oORAed4AaABAg,@jailtongiga,https://yt3.ggpht.com/b0M_lJEtjhRqpenO94EmyGx6...,http://www.youtube.com/@jailtongiga,UCJVjexEjkVqG6-YTEvzRvhA,Salve! gostaria contar minha história meu nome...,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
# Parece que a maioria dos dados ausentes são para esse comentário em específico
outros_sem_data = df.loc[(df['published_at'].isna()) & (df['comment_id'] != 'UgzNiQdoxnPl3oORAed4AaABAg')]
print(f"Quantidade: {len(outros_sem_data)}")

Quantidade: 1


In [31]:
# Além do caso anterior, esses 3 comentários também estão com dados faltando, mas mesmo para as informações preenchidas elas parecem erradas.
outros_sem_data.head()

,video_id,comment_id,author,author_profile_image_url,author_channel_url,author_channel_id,comment,published_at,updated_at,like_count,viewer_rating,can_rate,is_reply,parent_id,channel_id,language,language_langdetect
530169,JsXCe5O1Xr4,UgzEA24p6Uq-J0icvMJ4AaABAg,@allanpablo666,https://yt3.ggpht.com/R83SKZwInINagOmcYQzX0zuo...,http://www.youtube.com/@allanpablo666,UCA_tSXUwDUDyXU-oguhkHDQ,Difícil achar uma Oxandrolona boa procedência ...,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [32]:
df.dropna(inplace=True)

In [33]:
df.isna().sum()

video_id                    0
comment_id                  0
author                      0
author_profile_image_url    0
author_channel_url          0
author_channel_id           0
comment                     0
published_at                0
updated_at                  0
like_count                  0
viewer_rating               0
can_rate                    0
is_reply                    0
parent_id                   0
channel_id                  0
language                    0
language_langdetect         0
dtype: int64

In [34]:
df['viewer_rating'].value_counts()

viewer_rating
none    502679
Name: count, dtype: int64

In [35]:
# Shape final do df (525370, 17), ao todo 26.850 comentários foram removidos por conter dados faltantes
df.shape

(502679, 17)

In [36]:
# Data mínima e máxima de published_at
print('Data mínima de published_at:', df['published_at'].min())
print('Data máxima de published_at:', df['published_at'].max())

# Data mínima e máxima de updated_at
print('\nData mínima de updated_at:', df['updated_at'].min())
print('Data máxima de updated_at:', df['updated_at'].max())

Data mínima de published_at: 2020-01-01 21:42:44+00:00
Data máxima de published_at: 2024-12-31 23:56:29+00:00

Data mínima de updated_at: 2020-01-01 21:46:57+00:00
Data máxima de updated_at: 2024-12-31 23:56:29+00:00


In [37]:
# Único valor de viewer_rating é 'none', coluna não parece agregar muito.
print(df['viewer_rating'].value_counts())
df.drop(columns=['viewer_rating'], inplace=True)

viewer_rating
none    502679
Name: count, dtype: int64


---
## 4. Limpeza Final e Otimização

Remove colunas não informativas e duplicatas:
- **Colunas removidas**: viewer_rating (sempre 'none'), can_rate (sempre True)
- **Conversão de tipos**: like_count e is_reply convertidos para int
- **Deduplicação**: Remove comentários duplicados baseado em comment_id
- **Validação final**: Verifica contagens de vídeos, canais, comentários e usuários únicos
- **Output**: Dataset limpo salvo como `final_comments_match_cleaned.csv` (~525k comentários)

In [38]:
# Único valor de can_rate é True, coluna não parece agregar muito.
print(df['can_rate'].value_counts())
df.drop(columns=['can_rate'], inplace=True)

can_rate
True    502679
Name: count, dtype: int64


In [39]:
df['like_count'] = df['like_count'].astype(int)
df['is_reply'] = df['is_reply'].astype(int)

In [40]:
df.shape

(502679, 15)

In [41]:
df.drop_duplicates(subset='comment_id', inplace=True)

In [42]:
df.duplicated().sum()

np.int64(0)

In [43]:
df.head()

,video_id,comment_id,author,author_profile_image_url,author_channel_url,author_channel_id,comment,published_at,updated_at,like_count,is_reply,parent_id,channel_id,language,language_langdetect
0,g3YDGLa2pWg,UgzP9hlUh9w_IWQksvN4AaABAg,@dr.gabrielfeitosa,https://yt3.ggpht.com/yp_y0N8quVmXxwhO3S1rQqlX...,http://www.youtube.com/@dr.gabrielfeitosa,UCYgV9EbxVSNdDHBr0GWLCmQ,Segue uma playlist sobre colaterais de hormôni...,2024-12-31 23:07:02+00:00,2024-12-31 23:07:02+00:00,1,0,root,UCYgV9EbxVSNdDHBr0GWLCmQ,pt,rob_pt
66,g3YDGLa2pWg,UgyRKAO3TclHYnYDWmp4AaABAg,@wcbambam,https://yt3.ggpht.com/hDyV3pJYYSQzIIyDtAn723k4...,http://www.youtube.com/@wcbambam,UCsLYjxPkMIyHbWNOQP3lMNA,Excelente vídeo parabéns,2024-12-31 23:21:16+00:00,2024-12-31 23:21:16+00:00,0,0,root,UCYgV9EbxVSNdDHBr0GWLCmQ,pt,rob_pt
67,g3YDGLa2pWg,UgzHa9JnuuLqfPrOJGV4AaABAg,@wcbambam,https://yt3.ggpht.com/hDyV3pJYYSQzIIyDtAn723k4...,http://www.youtube.com/@wcbambam,UCsLYjxPkMIyHbWNOQP3lMNA,"Tenho 52 anos 1,80 de altura com 95 kg, faço a...",2024-12-31 23:17:34+00:00,2024-12-31 23:17:34+00:00,21,0,root,UCYgV9EbxVSNdDHBr0GWLCmQ,pt,rob_pt
68,g3YDGLa2pWg,UgzHa9JnuuLqfPrOJGV4AaABAg.ACjCRPAqgUmACjCt_J0G2S,@dr.gabrielfeitosa,https://yt3.ggpht.com/yp_y0N8quVmXxwhO3S1rQqlX...,http://www.youtube.com/@dr.gabrielfeitosa,UCYgV9EbxVSNdDHBr0GWLCmQ,Bacana seu relato. E que bom que você abandono...,2024-12-31 23:21:33+00:00,2024-12-31 23:21:33+00:00,7,1,UgzHa9JnuuLqfPrOJGV4AaABAg,UCYgV9EbxVSNdDHBr0GWLCmQ,pt,rob_pt
87,g3YDGLa2pWg,UgzUFvaS9843Qrg15ip4AaABAg,@pliniojose9485,https://yt3.ggpht.com/ytc/AIdro_mhD_uBtCD8mWdT...,http://www.youtube.com/@pliniojose9485,UC_kLXFVMk-ITuBzteT-WMrA,"Dr, o q o Sr acha de 1 ml/14 dias?",2024-12-31 23:05:37+00:00,2024-12-31 23:05:37+00:00,1,0,root,UCYgV9EbxVSNdDHBr0GWLCmQ,pt,rob_pt


In [44]:
print('Number of unique videos:', df['video_id'].nunique())
print('Number of unique channels:', df['channel_id'].nunique())
print('Number of unique comments:', df['comment_id'].nunique())
print('Number of unique users:', df['author_channel_id'].nunique())

Number of unique videos: 6742
Number of unique channels: 1551
Number of unique comments: 502677
Number of unique users: 216084


In [45]:
df.to_csv('cleaned_data/final_comments_match_cleaned.csv', index=False)